In [18]:
# Install dependencies
%pip install firebase-admin xgboost scikit-learn pandas numpy joblib schedule


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


How to firebase keyy??

Step by step to get Firebase service account key:

Go to: https://console.firebase.google.com

Click your existing project (the one with your 2 apps + website)

Click the ⚙️ gear icon (top left, next to project name)

Click "Project Settings"

Click the "Service Accounts" tab

Click the blue "Generate New Private Key" button

A JSON file downloads to your computer

In [ ]:
#Import and setup fire base no need to upload like in canva
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd
import numpy as np
import joblib
from datetime import datetime, timedelta
import schedule
import time
import os

def findFirebaseKey():
    startPath = os.path.abspath('../../')  # Go up 2 folders
    for root, dirs, files in os.walk(startPath):
        if 'healmind-2025-firebase-adminsdk-fbsvc-12242dbda6.json' in files:
            return os.path.join(root, 'healmind-2025-firebase-adminsdk-fbsvc-12242dbda6.json')
    raise FileNotFoundError("Firebase key not found!")

cred_path = findFirebaseKey()
cred = credentials.Certificate(cred_path)

try:
    firebase_admin.initialize_app(cred)
except ValueError:
    pass

db = firestore.client()
print(f"Firebase initialized! Found key at: {cred_path}")

Firebase initialized! Found key at: d:\laiba\Desktop\HealMind\healmind-2025-firebase-adminsdk-fbsvc-12242dbda6.json


In [11]:
import joblib
import os

# Find the model files
def findFiles(filename):
    startPath = os.path.abspath('../../')
    for root, dirs, files in os.walk(startPath):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"{filename} not found!")

# Load model
try:
    modelPath = findFiles('stress_model.pkl')
    model = joblib.load(modelPath)
    print(f"loaded model from {modelPath}")
except Exception as e:
    print(f"error loading model: {e}")
    model = None

# Load scaler
try:
    scalerPath = findFiles('scaler.pkl')
    scaler = joblib.load(scalerPath)
    print(f"loaded scaler from {scalerPath}")
except Exception as e:
    print(f"error loading scaler: {e}")
    scaler = None

if model and scaler:
    print("Files loaded successfully.")
else:
    print("Error: Model or scaler not loaded properly.")

loaded model from d:\laiba\Desktop\HealMind\HRV Module\XGBoost\stress_model.pkl
loaded scaler from d:\laiba\Desktop\HealMind\HRV Module\XGBoost\scaler.pkl
Files loaded successfully.


In [15]:
# Batch prediction class
class StressPredictor:
    def __init__(self, model, scaler, db):
        self.model = model
        self.scaler = scaler
        self.db = db
        self.feature_names = ['sdnn', 'rmssd']

    def fetch_unprocessed_data(self, hours=1):
        cutoff_time = datetime.utcnow() - timedelta(hours=hours)

        query = self.db.collection('heart_rate_data') \
            .where('timestamp', '>=', cutoff_time) \
            .stream()

        data_points = []
        for doc in query:
            data = doc.to_dict()
            data['doc_id'] = doc.id
            data_points.append(data)

        return pd.DataFrame(data_points) if data_points else pd.DataFrame()

    def group_by_window(self, df, window_minutes=5):
        if df.empty:
            return []

        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp')

        windows = []
        for i in range(0, len(df), window_minutes * 12):
            window = df.iloc[i:i+window_minutes*12]
            if len(window) >= 3:
                windows.append(window)

        return windows

    def process_window(self, window):
        all_ibi = []
        for ibi_list in window['ibi'].dropna():
            if ibi_list:
                all_ibi.extend(ibi_list)

        if not all_ibi or len(all_ibi) < 2:
            return None

        ibi = np.array(all_ibi, dtype=float)
        sdnn = np.std(ibi)
        rmssd = np.sqrt(np.mean(np.diff(ibi) ** 2))

        X = np.array([[sdnn, rmssd]])
        X_scaled = self.scaler.transform(X)

        prediction = self.model.predict(X_scaled)[0]
        probability = self.model.predict_proba(X_scaled)[0]

        return {
            'stress_level': int(prediction),
            'stress_probabilities': {
                'class_0': float(probability[0]),
                'class_1': float(probability[1]) if len(probability) > 1 else 0.0
            },
            'sdnn': float(sdnn),
            'rmssd': float(rmssd),
            'window_start': window['timestamp'].min(),
            'window_end': window['timestamp'].max(),
            'prediction_timestamp': datetime.utcnow(),
            'num_samples': len(window)
        }

    def store_predictions(self, results):
        batch = self.db.batch()

        for result in results:
            doc_ref = self.db.collection('stress_predictions').document()
            batch.set(doc_ref, result)

        batch.commit()
        return len(results)

    def run_batch(self, hours=1):
        print(f"\n{'='*60}")
        print(f"BATCH JOB: {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*60}")

        try:
            print(f"Fetching data from last {hours} hour(s)...")
            df = self.fetch_unprocessed_data(hours=hours)

            if df.empty:
                print("No new data to process")
                return

            print(f"Loaded {len(df)} data points")

            windows = self.group_by_window(df)
            print(f"Created {len(windows)} time windows")

            results = []
            for i, window in enumerate(windows):
                result = self.process_window(window)
                if result:
                    results.append(result)

            print(f"Processed {len(results)} windows")

            if results:
                stored = self.store_predictions(results)
                print(f"Stored {stored} predictions to Firestore")

                stress_high = sum(1 for r in results if r['stress_level'] == 1)
                avg_prob = np.mean([r['stress_probabilities']['class_1'] for r in results])
                print(f"\nSummary:")
                print(f"  High stress: {stress_high}/{len(results)}")
                print(f"  Avg probability: {avg_prob:.2%}")

            print(f"{'='*60}\n")
            return results

        except Exception as e:
            print(f"Error: {str(e)}")
            return None


In [17]:
# Initialize and run 
predictor = StressPredictor(model, scaler, db)

print("Testing batch processor...")
predictor.run_batch(hours=1)

Testing batch processor...

BATCH JOB: 2026-01-04 08:54:35
Fetching data from last 1 hour(s)...


d:\laiba\anaconda3\envs\TF\lib\site-packages\google\cloud\firestore_v1\base_collection.py:315: UserWarning: Detected filter using positional arguments. Prefer using the 'filter' keyword argument instead.
  return query.where(field_path, op_string, value)


No new data to process


In [16]:
# Schedule to run every 30 minutes
def job():
    predictor.run_batch(hours=1)

schedule.every(30).minutes.do(job)

print("Scheduler started!")
print("Will run predictions every 30 minutes")
print("\nKeep this cell running to continue processing...")

while True:
    schedule.run_pending()
    time.sleep(60)

Scheduler started!
Will run predictions every 30 minutes

Keep this cell running to continue processing...


KeyboardInterrupt: 